In [1]:
"""
查看鉴定结果的离子匹配情况：
    - 1电荷b离子、y离子存在匹配的概率
"""
import os
from utils import *

In [3]:
res_path = "D:/pFindWorkspace/Astral/new/top50-complete/result/restricted-0627/pFind-Filtered.spectra"  # 结果文件路径
data_dir = "D:/data/ms/Astral/PXD046357-Single_Cell/top50/"  # 质谱文件夹路径
mgf_paths = [os.path.join(data_dir,filename) for filename in os.listdir(data_dir) if filename.endswith(".mgf")][:3]

element2mass = get_element2mass("infos/element.ini")
aa2mass = get_aa2mass("infos/aa.ini", element2mass)
mod2mass = get_mod2mass("infos/modification.ini")

mgf_paths

['D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_01_20230808155512_HCDAST.mgf',
 'D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_02_20230808161140_HCDAST.mgf',
 'D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_03_20230808162807_HCDAST.mgf']

In [25]:
spec2res = dict()
f = open(res_path,"r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    spectrum_name = items[0]
    peptide = items[5]+'\t'+items[10]  # seq+mod
    is_decoy = (items[15]=="decoy")
    
    proteins = items[12].split("/")[:-1]
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag or is_decoy:
        continue
    spec2res[spectrum_name] = peptide
f.close()


i = 0
res = list()
for mgf_path in mgf_paths:
    print(mgf_path)
    f = open(mgf_path,"r")
    while True:
        line = f.readline()
        if not line:
            break
        if line.startswith("T"):
            spectrum_name = line.strip()[6:]
            i += 1
            if i%10000 == 0:
                print(i)
        elif line.startswith("C"):
            charge = int(line.strip()[7:-1])
        elif 'A'<=line[0]<='Z':
            continue
        else:
            peaks = list()
            flag = (spectrum_name in spec2res)
            while line[0]!='E':
                if flag:
                    items = line.strip().split()
                    mz,inten = float(items[0]),float(items[1])
                    peaks.append([mz,inten])
                line = f.readline()

            if flag:
                res.append(cal_by_matches(peaks, spec2res[spectrum_name], charge, aa2mass, mod2mass, topk=200))
    f.close()

D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_01_20230808155512_HCDAST.mgf
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_02_20230808161140_HCDAST.mgf
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
640000
650000
660000
670000
680000
690000
700000
710000
720000
730000
740000
750000
760000
770000
780000
790000
800000
D:/data/ms/Astral/PXD046357-Single_Cell/top50/20230108_AST_Neo1_DDA_UHG_HeLa_200ng_2th2p5ms_top50_03_20230808162807_HCDAST.mgf
810000
820000
830000
840000
850000
860000
870000
880000
890000
90000

In [49]:
b = 0
y = 0
by = 0
for matches in res:
    flag_b = False
    flag_y = False
    for match in matches:
        if match[0][0] == 'b':
            flag_b = True
        elif match[0][0] == 'y':
            flag_y = True
    # print(match)
    # break
    if not flag_b:
        b += 1
    if not flag_y:
        print(matches)
        y += 1
        # break
    if not flag_b and not flag_y:
        by += 1
print(b,y,by, len(res))

[]
[('b2+', 195.087006, 13545.3, 3.270335698075139e-06), ('b3+', 282.119965, 4027.3, -1.0513258074906497e-06), ('b4+', 445.181305, 2708.0, 3.7876702841208525e-06), ('b7+', 760.287537, 3627.4, 2.749880667915516e-06), ('b8+', 859.355652, 11107.0, 2.7746370138452533e-06), ('b9++', 498.71048, 6182.1, 3.26772760015779e-06), ('b10++', 542.228271, 3718.9, -2.750686525094345e-07), ('b11++', 570.738708, 6653.8, 2.530054437230058e-07), ('b12++', 606.256897, 5504.1, 8.421347494512284e-07)]
[('b2+', 171.11203, 1849.4, 4.4765993366960355e-06), ('b3+', 284.194733, 1118.6, 7.462840629149273e-06)]
[('b2+', 228.145569, 1812.0, -3.4890004820000975e-07), ('b6+', 582.334717, 3187.6, 1.8465325331765653e-06), ('b7+', 696.376099, 3174.4, 3.7558727301478855e-06), ('b13++', 676.308899, 3672.0, 4.861979496042415e-06)]
[('b3+', 240.13385, 2813.9, 1.6953045146200346e-06), ('b4+', 339.203125, 2091.8, -1.3537611128026345e-06), ('b13++', 620.298096, 10884.1, -3.0629628112515506e-06)]
[('b2+', 228.133987, 3569.3, 1.1

In [61]:
cal_precursor_mh("NLRSETEEILNENIDK", aa2mass)

1916.9558836

In [63]:
(1916.9558836+0.9840*2)/2 + 1

960.4619418000001